### Classify critera to 4 categories

In [ ]:
import json
import os
from vertexai.generative_models import GenerativeModel
from google.oauth2 import service_account
import vertexai

In [ ]:


# Get credentials path from environment or use default
credentials_path = os.environ.get(
    "VERTEXAI_CREDENTIALS_JSON_PATH", 
    "vertex-creds.json"
)
credentials = service_account.Credentials.from_service_account_file(credentials_path)

# Initialize Vertex AI with project from credentials
vertexai.init(project=credentials.project_id, credentials=credentials)

# Return the initialized model
model = GenerativeModel('gemini-2.5-pro')  # gemini-2.5-flash 



# Classify criteria

In [ ]:
with open('./grading_criteria.txt', 'r') as file:
    criteria = file.readlines()
    criteria = [line.strip() for line in criteria]  # Remove newline characte

In [ ]:
len(criteria)

In [ ]:
prompt = f"""You are provided a list of criteria to evaluate EIS section of NEPA draft and you are required to classify them into one of the four classes:
Structure: This is criteria that check if the draft has all required component and follows a structure or not. This is criteria only checks about organization of draft nothing else, usually has structure or organization in the criteria name.
Clarity: This criteria checks if the draft if the tone of the draft is objective, formal and non-speculative.
Accuracy: This criteria checks accuracy of the generated draft for multiple requirements. This can be different aspect a particular EIS document of NEPA draft.
Reference: Checks if relevant references are provided. 

Given list of criteria {criteria}

Strictly return a json with all criteria as key and one of the four key as value.
"""



In [ ]:
result = model.generate_content(prompt)

In [ ]:
classification_result = result.candidates[0].content.parts[0].text

In [ ]:
classification_result

In [ ]:

def parse_escaped_json_string(escaped_json_str):
    """
    Converts a string containing escaped JSON and optional markdown formatting into a Python dictionary,
    preserving special characters like en dashes.
    """
    # Step 1: Remove markdown formatting if present
    if escaped_json_str.startswith("```json"):
        escaped_json_str = escaped_json_str.replace("```json", "").strip("`")

    # Step 2: Decode escape sequences safely
    try:
        # First decode escape sequences
        unescaped_str = bytes(escaped_json_str, "utf-8").decode("unicode_escape")

        # Then re-encode and decode as UTF-8 to fix characters like en dash
        fixed_str = unescaped_str.encode("latin1").decode("utf-8")
    except Exception as e:
        print("Decoding error:", e)
        return None

    # Step 3: Parse JSON
    try:
        return json.loads(fixed_str)
    except json.JSONDecodeError as e:
        print("Failed to parse JSON:", e)
        print("Final string was:", repr(fixed_str))
        return None


In [ ]:
result_json = parse_escaped_json_string(classification_result)

In [ ]:
len(result_json)

In [ ]:
result_json

In [ ]:
# check if all criteria present 
for crit in criteria:
    if not result_json[crit]:
        print(crit)

In [ ]:
# save json 
with open("criteria_classes.json", "w", encoding="utf-8") as f:
    json.dump(result_json, f, ensure_ascii=False, indent=2)
    print(f"JSON saved successfully to {result_json}")
